# 🏛️ Asset-Class Backtest — Long History (1970s and earlier)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/free-portfolio-visualizer/blob/main/notebooks/08_asset_class_history.ipynb)

Portfolio Visualizer's asset-class backtests start in 1972. This notebook builds the same style of long-history series from free academic/government data — Ken French equity portfolios (1926+), synthetic Treasury returns from FRED yields, T-bill cash, gold — and backtests allocations across decades of regimes. These are research proxies, not investable fund histories.

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/free-portfolio-visualizer.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Allocation {display-mode: "form"}
#@markdown Pick from: US Stock Market, US Large Cap Value, US Large Cap Growth, US Small Cap, US Small Cap Value, US Small Cap Growth, Long Term Treasury, 10-Year Treasury, Intermediate Treasury, Short Term Treasury, Cash (T-Bills), Gold
asset_classes = "US Stock Market, Long Term Treasury, Gold"  #@param {type:"string"}
weights = "60, 30, 10"       #@param {type:"string"}
start_year = 1972            #@param {type:"number"}
initial_amount = 10000       #@param {type:"number"}
rebalancing = "annually"     #@param ["never", "monthly", "quarterly", "annually"]
SMOKE = False

In [ ]:
import pandas as pd
from portlab import backtest_portfolio, metrics, plots
from portlab.data.asset_classes import get_asset_class_returns

classes = [c.strip() for c in asset_classes.split(",") if c.strip()]
w = dict(zip(classes, [float(x) for x in weights.split(",")]))
rets = get_asset_class_returns(classes, start=f"{start_year}-01-01").dropna()
print(f"{rets.index.min():%Y-%m} → {rets.index.max():%Y-%m}  ({len(rets)} months)")

result = backtest_portfolio(rets, w, initial=initial_amount,
                            rebalance=rebalancing, periods=12)
result.summary().style.format("{:.4f}", na_rep="—")

In [ ]:
plots.growth_chart(result.returns, initial=initial_amount, log_scale=True).show()
plots.drawdown_chart(result.returns).show()
plots.annual_returns_chart(result.returns).show()
print("Worst drawdowns across the full history:")
result.drawdowns()

In [ ]:
# Dynamic allocation (PV's backtest-dynamic-allocation): change weights over time
from portlab.backtest import backtest_dynamic
schedule = {
    f"{start_year}-01-01": {classes[0]: 0.9, classes[-1]: 0.1},   # aggressive early
    f"{min(start_year + 25, 2015)}-01-01": {c: 1/len(classes) for c in classes},  # de-risk later
}
dyn = backtest_dynamic(rets, schedule, initial=initial_amount, periods=12)
both = pd.concat([result.returns.rename("Static"), dyn.returns.rename("Dynamic")], axis=1)
plots.growth_chart(both, initial=initial_amount, log_scale=True).show()
plots.weights_chart(dyn.weights, "Dynamic Allocation Weights").show()